# Training — SeribuCerita Emotion Classifier
**Capstone Project CC26-PSU212 — AI Path**


## 1. Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q tensorflow
!pip install -q "transformers==4.44.0"
!pip install -q tf-keras
!pip install -q scikit-learn pandas numpy matplotlib seaborn tqdm

# Verifikasi
import transformers, tensorflow as tf
print(f"transformers version : {transformers.__version__}")
print(f"tensorflow version   : {tf.__version__}")
print(f"GPU available        : {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"GPU devices          : {tf.config.list_physical_devices('GPU')}")

# Verify TFBertModel importable
from transformers import TFBertModel, BertTokenizer
from transformers.utils import is_tf_available
print(f"\nTFBertModel importable: {TFBertModel is not None}")
print(f"is_tf_available()      : {is_tf_available()}")
print("\nAll dependencies ready ✓")

In [ ]:
# Mount Google Drive (Colab) or set local output directory
# Uncomment the following lines if running on Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

import os
OUTPUT_DIR = './output'  # Change to your preferred output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_recall_fscore_support
)

os.environ['TF_USE_LEGACY_KERAS'] = '1'

from transformers import BertTokenizer, TFBertModel

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Display
pd.set_option('display.max_colwidth', 100)
plt.rcParams['figure.dpi'] = 100

print("All imports successful ✓")
print(f"tensorflow  : {tf.__version__}")
print(f"transformers: {__import__('transformers').__version__}")

## 2. Configuration
Semua hyperparameter terkonsentrasi di sini untuk memudahkan tuning nanti.

In [ ]:
# ==========================================
# CONFIG V4 — tuned hyperparameters
# ==========================================
class Config:
    # ===== Data =====
    TRAIN_PATH = '../data/train_5kls.csv'
    VALID_PATH = '../data/valid_5kls.csv'
    TEST_PATH  = '../data/test_5kls.csv'
    TEXT_COL   = 'tweet'
    LABEL_COL  = 'label_id'
    # ===== Model =====
    MODEL_NAME    = 'indobenchmark/indobert-base-p1'
    NUM_CLASSES   = 5
    MAX_LENGTH    = 128
    FREEZE_LAYERS = 4
    # ===== Training =====
    BATCH_SIZE    = 16
    EPOCHS        = 12
    BASE_LR       = 1.5e-5
    HEAD_LR_MULT  = 10.0
    LAYER_DECAY   = 0.9
    WEIGHT_DECAY  = 0.01
    WARMUP_RATIO  = 0.1
    GRAD_CLIP     = 1.0
    DROPOUT       = 0.2
    # ===== Loss =====
    FOCAL_GAMMA       = 2.0
    LABEL_SMOOTHING   = 0.05
    RDROP_ALPHA       = 0.5
    # ===== Callback =====
    EARLY_STOP_PATIENCE = 5
    # ===== Output (Google Drive) =====
    DRIVE_DIR      = OUTPUT_DIR
    LOG_DIR        = f'{DRIVE_DIR}/logs/run_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    MODEL_KERAS    = f'{DRIVE_DIR}/model_final.keras'
    SAVED_MODEL    = f'{DRIVE_DIR}/saved_model'
    TOKENIZER_DIR  = f'{DRIVE_DIR}/tokenizer'

cfg = Config()
print(f'Log directory: {cfg.LOG_DIR}')
os.makedirs(cfg.LOG_DIR, exist_ok=True)
print(f'All outputs will be saved to: {cfg.DRIVE_DIR}')


## 3. Load Data
Load dari CSV yang dihasilkan notebook dataset prep.

In [ ]:
# Load splits
df_train = pd.read_csv(cfg.TRAIN_PATH)
df_valid = pd.read_csv(cfg.VALID_PATH)
df_test  = pd.read_csv(cfg.TEST_PATH)

print(f"Train: {len(df_train)} rows")
print(f"Valid: {len(df_valid)} rows")
print(f"Test : {len(df_test)} rows")

print("\n=== TRAIN LABEL DISTRIBUTION ===")
print(df_train['label'].value_counts())
print("\n=== TRAIN LABEL_ID DISTRIBUTION ===")
print(df_train['label_id'].value_counts().sort_index())

# Label mapping verification
LABEL2ID = {'anger': 0, 'fear': 1, 'sad': 2, 'neutral': 3, 'happy': 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print("\nLabel mapping:", LABEL2ID)

## 4. Class Weights

Compute inverse-frequency class weights to handle label imbalance in the loss function.

In [ ]:
# Compute class weights (balanced inverse frequency)
y_train = df_train[cfg.LABEL_COL].values

class_weights = compute_class_weight(
    'balanced',
    classes=np.arange(cfg.NUM_CLASSES),
    y=y_train
)

print("=== CLASS WEIGHTS ===")
for label_id, weight in enumerate(class_weights):
    label_name = ID2LABEL[label_id]
    n_samples = (y_train == label_id).sum()
    print(f"  {label_id} ({label_name:8s}): weight={weight:.4f}, n={n_samples}")

print(f"\nClass weights array: {class_weights}")
class_weights_tf = tf.constant(class_weights, dtype=tf.float32)

## 5. Tokenization
Pakai IndoBERT tokenizer. Hasil tokenize disimpan dalam `tf.data.Dataset` untuk efficient batching.

In [ ]:
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained(cfg.MODEL_NAME)
print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")
print(f"Max length: {cfg.MAX_LENGTH}")

# Save tokenizer untuk inference nanti
os.makedirs(cfg.TOKENIZER_DIR, exist_ok=True)
tokenizer.save_pretrained(cfg.TOKENIZER_DIR)
print(f"Tokenizer saved to {cfg.TOKENIZER_DIR}")

In [ ]:
def tokenize_texts(texts, tokenizer, max_length):
    """Tokenize a list of texts into BERT input format."""
    encoded = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='tf'
    )
    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
    }

print("Tokenizing training set...")
train_enc = tokenize_texts(df_train[cfg.TEXT_COL].astype(str).tolist(), tokenizer, cfg.MAX_LENGTH)
print(f"  input_ids shape : {train_enc['input_ids'].shape}")
print(f"  attention_mask shape: {train_enc['attention_mask'].shape}")

print("\nTokenizing validation set...")
valid_enc = tokenize_texts(df_valid[cfg.TEXT_COL].astype(str).tolist(), tokenizer, cfg.MAX_LENGTH)

print("\nTokenizing test set...")
test_enc = tokenize_texts(df_test[cfg.TEXT_COL].astype(str).tolist(), tokenizer, cfg.MAX_LENGTH)

print("\nDone tokenization")

In [ ]:
# Build tf.data.Dataset
def make_dataset(encodings, labels, batch_size, shuffle=False, buffer_size=10000):
    inputs = {
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
    }
    labels = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((inputs, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size, seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_enc, df_train[cfg.LABEL_COL].values, cfg.BATCH_SIZE, shuffle=True)
valid_ds = make_dataset(valid_enc, df_valid[cfg.LABEL_COL].values, cfg.BATCH_SIZE, shuffle=False)
test_ds  = make_dataset(test_enc,  df_test[cfg.LABEL_COL].values,  cfg.BATCH_SIZE, shuffle=False)

n_train_steps = len(df_train) // cfg.BATCH_SIZE + 1
n_valid_steps = len(df_valid) // cfg.BATCH_SIZE + 1
print(f"Train batches: {n_train_steps}, Valid batches: {n_valid_steps}")

## 6. AttentionPoolingLayer

Learned attention pooling over BERT hidden states, replacing the default `[CLS]` token pooling.

In [ ]:
class AttentionPoolingLayer(tf.keras.layers.Layer):
    """
    Custom attention pooling untuk menggantikan [CLS] token pooling default BERT.
    Cara kerja:
    1. Hitung attention score untuk setiap token
    2. Mask out padding tokens (set score ke -infinity)
    3. Softmax untuk dapat attention weights
    4. Weighted sum dari hidden states
    Reference: Lin et al. (2017) "A Structured Self-attentive Sentence Embedding"
    """
    def __init__(self, hidden_size=768, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.attention_dense = tf.keras.layers.Dense(1, name='attention_dense')

    def call(self, hidden_states, attention_mask):
        # hidden_states: [batch, seq_len, hidden]
        # attention_mask: [batch, seq_len]
        # Compute attention scores
        scores = self.attention_dense(hidden_states)              # [batch, seq_len, 1]
        scores = tf.squeeze(scores, axis=-1)                      # [batch, seq_len]
        # Mask out padding tokens
        mask = tf.cast(attention_mask, tf.float32)
        scores = scores + (1.0 - mask) * -1e9                     # padding -> -inf
        # Softmax to get attention weights
        weights = tf.nn.softmax(scores, axis=-1)                  # [batch, seq_len]
        weights = tf.expand_dims(weights, axis=-1)                # [batch, seq_len, 1]
        # Weighted sum
        pooled = tf.reduce_sum(hidden_states * weights, axis=1)   # [batch, hidden]
        return pooled
    def get_config(self):
        config = super().get_config()
        config.update({'hidden_size': self.hidden_size})
        return config

# Quick test
test_layer = AttentionPoolingLayer(hidden_size=768)
dummy_hidden = tf.random.normal((2, 10, 768))
dummy_mask = tf.constant([[1,1,1,1,1,0,0,0,0,0], [1,1,1,1,1,1,1,0,0,0]], dtype=tf.int32)
out = test_layer(dummy_hidden, dummy_mask)
print(f"AttentionPoolingLayer test output shape: {out.shape} (expected: (2, 768))")
assert out.shape == (2, 768)
print("✓ AttentionPoolingLayer works correctly")

## 7. EmotionClassifier (Model Subclassing)

IndoBERT backbone → AttentionPoolingLayer → Dropout → Dense classifier.

In [ ]:
class EmotionClassifier(tf.keras.Model):
    def __init__(self, model_name, num_classes, dropout=0.3, **kwargs):
        super().__init__(**kwargs)
        self.model_name = model_name
        self.num_classes = num_classes
        self.dropout_rate = dropout
        self.bert = TFBertModel.from_pretrained(model_name, name='bert')
        self.pooler = AttentionPoolingLayer(hidden_size=768, name='attention_pooler')
        self.dropout = tf.keras.layers.Dropout(dropout, name='dropout')
        self.classifier = tf.keras.layers.Dense(num_classes, name='classifier')
    def call(self, inputs, training=False):
        bert_out = self.bert(input_ids=inputs['input_ids'],
                            attention_mask=inputs['attention_mask'], training=training)
        pooled = self.pooler(bert_out.last_hidden_state, inputs['attention_mask'])
        pooled = self.dropout(pooled, training=training)
        return self.classifier(pooled)

print('Building model...')
model = EmotionClassifier(cfg.MODEL_NAME, cfg.NUM_CLASSES, cfg.DROPOUT)
dummy_input = {'input_ids': tf.zeros((1, cfg.MAX_LENGTH), dtype=tf.int32),
               'attention_mask': tf.ones((1, cfg.MAX_LENGTH), dtype=tf.int32)}
_ = model(dummy_input, training=False)

# Freeze bottom BERT layers
model.bert.bert.embeddings.trainable = False
for i, layer in enumerate(model.bert.bert.encoder.layer):
    if i < cfg.FREEZE_LAYERS:
        layer.trainable = False
if hasattr(model.bert.bert, 'pooler') and model.bert.bert.pooler is not None:
    model.bert.bert.pooler.trainable = False

tp = sum(np.prod(v.shape) for v in model.trainable_variables)
ap = sum(np.prod(v.shape) for v in model.variables)
print(f'Trainable: {tp:,} / {ap:,} ({tp/ap*100:.1f}%)')
print(f'Frozen: embeddings + layers 0-{cfg.FREEZE_LAYERS-1} + pooler')


## 8. FocalLossWithSmoothing

Custom loss combining focal weighting (γ=2.0), class weights, and label smoothing.

In [ ]:
class FocalLossWithSmoothing(tf.keras.losses.Loss):
    """
    Custom loss function: Focal Loss + Class Weights + Label Smoothing.
    Components:
    - Focal weighting: down-weight easy examples, focus on hard ones
    - Class weights: handle class imbalance
    - Label smoothing: regularization against overconfident predictions
                       (especially helpful for noisy labels)
    Reference: Lin et al. (2017) "Focal Loss for Dense Object Detection"
    """
    def __init__(self, num_classes, gamma=2.0, label_smoothing=0.1,
                 class_weights=None, name='focal_loss_with_smoothing', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        if class_weights is not None:
            self.class_weights = tf.constant(class_weights, dtype=tf.float32)
        else:
            self.class_weights = None
    def call(self, y_true, y_pred):
        # y_true: [batch] integer labels
        # y_pred: [batch, num_classes] logits
        y_true_int = tf.cast(y_true, tf.int32)
        y_true_oh  = tf.one_hot(y_true_int, depth=self.num_classes)
        # Apply label smoothing
        smooth = self.label_smoothing
        y_true_smooth = y_true_oh * (1.0 - smooth) + smooth / self.num_classes
        # Convert logits to probabilities
        probs = tf.nn.softmax(y_pred, axis=-1)
        probs = tf.clip_by_value(probs, 1e-7, 1.0 - 1e-7)
        # Cross-entropy with smoothed labels
        ce = -y_true_smooth * tf.math.log(probs)              # [batch, num_classes]
        # Focal weighting: (1 - p_t)^gamma
        focal_weight = tf.pow(1.0 - probs, self.gamma)        # [batch, num_classes]
        focal_ce = focal_weight * ce                           # [batch, num_classes]
        # Sum over classes
        per_sample = tf.reduce_sum(focal_ce, axis=-1)         # [batch]
        # Apply class weights per sample
        if self.class_weights is not None:
            sample_weights = tf.gather(self.class_weights, y_true_int)
            per_sample = per_sample * sample_weights
        return tf.reduce_mean(per_sample)
    def get_config(self):
        config = super().get_config()
        config.update({
            'num_classes': self.num_classes,
            'gamma': self.gamma,
            'label_smoothing': self.label_smoothing,
        })
        return config

# Build loss
loss_fn = FocalLossWithSmoothing(
    num_classes=cfg.NUM_CLASSES,
    gamma=cfg.FOCAL_GAMMA,
    label_smoothing=cfg.LABEL_SMOOTHING,
    class_weights=class_weights,
)
print(f"Loss function ready: {loss_fn.name}")
print(f"  gamma          : {cfg.FOCAL_GAMMA}")
print(f"  label_smoothing: {cfg.LABEL_SMOOTHING}")
print(f"  class_weights  : {class_weights.round(3).tolist()}")

# Quick test
dummy_logits = tf.random.normal((4, 5))
dummy_labels = tf.constant([0, 1, 2, 3], dtype=tf.int32)
test_loss = loss_fn(dummy_labels, dummy_logits)
print(f"\nTest loss value: {test_loss.numpy():.4f}")

## 9. Layer-wise Learning Rate Decay

Lower layers get smaller LR (preserve pretrained representations), upper layers and classifier head get larger LR.

In [ ]:
def get_layerwise_lr_groups(model, base_lr, layer_decay, head_lr_mult):
    groups = []
    bert = model.bert
    n = len(bert.bert.encoder.layer)
    ev = bert.bert.embeddings.trainable_variables
    if ev:
        groups.append({'name':'embeddings','vars':ev,'lr':base_lr*(layer_decay**(n+1))})
    for i, layer in enumerate(bert.bert.encoder.layer):
        lv = layer.trainable_variables
        if lv:
            groups.append({'name':f'layer_{i}','vars':lv,'lr':base_lr*(layer_decay**(n-i))})
    head_vars = model.pooler.trainable_variables + model.classifier.trainable_variables
    groups.append({'name':'head','vars':head_vars,'lr':base_lr*head_lr_mult})
    return groups

groups = get_layerwise_lr_groups(model, cfg.BASE_LR, cfg.LAYER_DECAY, cfg.HEAD_LR_MULT)
print('=== LR GROUPS ===')
for g in groups:
    nv = sum(np.prod(v.shape) for v in g['vars'])
    print(f"  {g['name']:15s}: lr={g['lr']:.2e}, params={nv:>10,}")


## 10. Optimizer Setup — AdamW dengan Layer-wise LR
Bikin satu AdamW per LR group. Weight decay 0.01 (BERT paper standard).

In [ ]:
# Create one AdamW optimizer per LR group
# Each optimizer manages its own variables with its specific LR
optimizers = []
for g in groups:
    opt = tf.keras.optimizers.AdamW(
        learning_rate=g['lr'],
        weight_decay=cfg.WEIGHT_DECAY,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-8,
    )
    optimizers.append(opt)

print(f"Created {len(optimizers)} optimizers (one per LR group)")
print(f"AdamW weight_decay: {cfg.WEIGHT_DECAY}")
print("\nNote: Optimizers will be initialized via a warmup step before training (see Section 14).")

## 11. Learning Rate Schedule — Linear Warmup + Linear Decay
Selama warmup (10% awal): LR naik dari 0 ke LR target masing-masing group.
Setelah warmup: LR turun linear ke 0 sampai akhir training.

In [ ]:
# Compute total steps and warmup steps
total_steps = n_train_steps * cfg.EPOCHS
warmup_steps = int(total_steps * cfg.WARMUP_RATIO)

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps} ({cfg.WARMUP_RATIO*100:.0f}%)")
print(f"Decay steps  : {total_steps - warmup_steps}")

# Store initial LRs for scheduling
initial_lrs = [g['lr'] for g in groups]

def update_learning_rates(step, total_steps, warmup_steps, initial_lrs, optimizers):
    """Update LR for each optimizer based on current step (linear warmup + linear decay)."""
    if step < warmup_steps:
        # Linear warmup: 0 -> initial_lr
        scale = (step + 1) / max(1, warmup_steps)
    else:
        # Linear decay: initial_lr -> 0
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        scale = max(0.0, 1.0 - progress)
    for opt, init_lr in zip(optimizers, initial_lrs):
        opt.learning_rate.assign(init_lr * scale)
    return scale

print("\nLR schedule function ready")

## 12. F1ScoreCallback

Monitors macro F1 on validation set for early stopping and best-weight restoration.

In [ ]:
class F1ScoreCallback:
    """
    Custom callback untuk monitor macro F1 di validation.
    Lebih sesuai untuk imbalanced multi-class classification.
    Features:
    - Early stopping berdasarkan macro F1 (bukan val_loss)
    - Restore best weights setelah training selesai
    - TensorBoard logging
    """
    def __init__(self, model, valid_dataset, patience=3, log_dir=None,
                 id2label=None):
        self.model = model
        self.valid_dataset = valid_dataset
        self.patience = patience
        self.id2label = id2label or {}
        self.best_f1 = -1.0
        self.best_epoch = -1
        self.best_weights = None
        self.wait = 0
        self.stop_training = False
        self.history = []
        # TensorBoard writer
        self.writer = None
        if log_dir:
            self.writer = tf.summary.create_file_writer(f"{log_dir}/valid_metrics")
    def on_epoch_end(self, epoch):
        # Predict on validation set
        y_true, y_pred = [], []
        for batch_x, batch_y in self.valid_dataset:
            logits = self.model(batch_x, training=False)
            preds = tf.argmax(logits, axis=-1).numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(batch_y.numpy().tolist())
        # Compute metrics
        macro_f1 = f1_score(y_true, y_pred, average='macro')
        weighted_f1 = f1_score(y_true, y_pred, average='weighted')
        acc = accuracy_score(y_true, y_pred)
        # Per-class F1
        prec, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=None, labels=list(range(len(self.id2label)))
        )
        # Save history
        self.history.append({
            'epoch': epoch,
            'val_acc': acc,
            'val_macro_f1': macro_f1,
            'val_weighted_f1': weighted_f1,
            'per_class_f1': f1.tolist(),
        })
        # Print
        print(f"\n  Validation: acc={acc:.4f}, macro_f1={macro_f1:.4f}, weighted_f1={weighted_f1:.4f}")
        for i, (p, r, f) in enumerate(zip(prec, recall, f1)):
            label_name = self.id2label.get(i, f'class_{i}')
            print(f"    {label_name:10s}: P={p:.3f}, R={r:.3f}, F1={f:.3f}")
        # TensorBoard logging
        if self.writer is not None:
            with self.writer.as_default():
                tf.summary.scalar('val_accuracy', acc, step=epoch)
                tf.summary.scalar('val_macro_f1', macro_f1, step=epoch)
                tf.summary.scalar('val_weighted_f1', weighted_f1, step=epoch)
                for i, f in enumerate(f1):
                    label_name = self.id2label.get(i, f'class_{i}')
                    tf.summary.scalar(f'val_f1_{label_name}', f, step=epoch)
                self.writer.flush()
        # Check improvement
        if macro_f1 > self.best_f1:
            improvement = macro_f1 - self.best_f1
            self.best_f1 = macro_f1
            self.best_epoch = epoch
            self.wait = 0
            # Save best weights (in memory)
            self.best_weights = [w.numpy() for w in self.model.trainable_variables]
            print(f"  ✓ Best macro_f1 improved by {improvement:+.4f} → {macro_f1:.4f}")
        else:
            self.wait += 1
            print(f"  ✗ No improvement ({self.wait}/{self.patience})")
            if self.wait >= self.patience:
                self.stop_training = True
                print(f"  ⏹ Early stopping triggered. Best macro_f1: {self.best_f1:.4f} (epoch {self.best_epoch+1})")
    def restore_best_weights(self):
        """Restore weights yang paling bagus ke model."""
        if self.best_weights is not None:
            for var, best_val in zip(self.model.trainable_variables, self.best_weights):
                var.assign(best_val)
            print(f"✓ Restored best weights from epoch {self.best_epoch+1} (macro_f1={self.best_f1:.4f})")

# Instantiate
f1_callback = F1ScoreCallback(
    model=model,
    valid_dataset=valid_ds,
    patience=cfg.EARLY_STOP_PATIENCE,
    log_dir=cfg.LOG_DIR,
    id2label=ID2LABEL,
)
print(f"F1ScoreCallback ready (patience={cfg.EARLY_STOP_PATIENCE})")

## 13. Custom Training Loop

Manual training loop with `tf.GradientTape`, R-Drop regularization, gradient clipping, and per-group optimizer updates.

In [ ]:
# R-Drop: Liang et al. 2021 — double forward pass + KL divergence
@tf.function
def train_step(model, batch, loss_fn, optimizers, var_groups, grad_clip, rdrop_alpha=1.0):
    x, y = batch
    with tf.GradientTape() as tape:
        logits1 = model(x, training=True)
        logits2 = model(x, training=True)
        ce_loss = (loss_fn(y, logits1) + loss_fn(y, logits2)) / 2.0
        p = tf.clip_by_value(tf.nn.softmax(logits1, axis=-1), 1e-7, 1.0)
        q = tf.clip_by_value(tf.nn.softmax(logits2, axis=-1), 1e-7, 1.0)
        kl = tf.reduce_mean(tf.reduce_sum(p * tf.math.log(p/q), axis=-1) +
                            tf.reduce_sum(q * tf.math.log(q/p), axis=-1)) / 2.0
        loss = ce_loss + rdrop_alpha * kl
    all_vars = []
    for gv in var_groups: all_vars.extend(gv)
    grads = tape.gradient(loss, all_vars)
    grads, _ = tf.clip_by_global_norm(grads, clip_norm=grad_clip)
    idx = 0
    for opt, gv in zip(optimizers, var_groups):
        n = len(gv)
        opt.apply_gradients(zip(grads[idx:idx+n], gv))
        idx += n
    preds = tf.argmax(logits1, axis=-1)
    acc = tf.reduce_mean(tf.cast(tf.equal(preds, tf.cast(y, tf.int64)), tf.float32))
    return loss, acc

@tf.function
def valid_step(model, batch, loss_fn):
    x, y = batch
    logits = model(x, training=False)
    loss = loss_fn(y, logits)
    preds = tf.argmax(logits, axis=-1)
    acc = tf.reduce_mean(tf.cast(tf.equal(preds, tf.cast(y, tf.int64)), tf.float32))
    return loss, acc

print('R-Drop train_step ready')


In [ ]:
# Extract var groups (just the vars, in order)
var_groups_only = [g['vars'] for g in groups]

# TensorBoard writers
train_writer = tf.summary.create_file_writer(f"{cfg.LOG_DIR}/train")

print(f"TensorBoard logs: {cfg.LOG_DIR}")
print(f"Run this to launch TB later: %tensorboard --logdir {cfg.LOG_DIR}")

## 14. Main Training Loop
Now we put it all together. Estimated time: ~7-12 menit per epoch di T4 GPU.

In [ ]:
import time

# ===== OPTIMIZER WARMUP STEP =====
# IMPORTANT: Keras 3 with multiple optimizers requires variables to be registered
# via at least one apply_gradients call OUTSIDE @tf.function before using @tf.function.
# We do a dummy gradient step (zero gradients) to trigger this registration.
print("Performing optimizer warmup (registers variables with optimizers)...")

# Get one batch for warmup
warmup_batch = next(iter(train_ds))
warmup_x, warmup_y = warmup_batch

with tf.GradientTape() as tape:
    logits = model(warmup_x, training=True)
    loss = loss_fn(warmup_y, logits)

# Collect all vars
all_vars_warmup = []
for g_vars in var_groups_only:
    all_vars_warmup.extend(g_vars)

# Compute and apply gradients (this registers vars with optimizers)
grads = tape.gradient(loss, all_vars_warmup)
grads, _ = tf.clip_by_global_norm(grads, clip_norm=cfg.GRAD_CLIP)

idx = 0
for opt, g_vars in zip(optimizers, var_groups_only):
    n = len(g_vars)
    opt.apply_gradients(zip(grads[idx:idx+n], g_vars))
    idx += n

print(f"Warmup complete. Initial loss: {loss.numpy():.4f}")
print(f"All optimizers initialized: {len(optimizers)} optimizers ready")

global_step = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)
print(f"Epochs        : {cfg.EPOCHS} (max)")
print(f"Batch size    : {cfg.BATCH_SIZE}")
print(f"Total steps   : {total_steps}")
print(f"Warmup steps  : {warmup_steps}")
print(f"Early stop    : patience {cfg.EARLY_STOP_PATIENCE} on macro_f1")
print("=" * 70)

epoch_start_time = time.time()

for epoch in range(cfg.EPOCHS):
    print(f"\n{'='*70}\nEpoch {epoch+1}/{cfg.EPOCHS}\n{'='*70}")
    epoch_start_time = time.time()
    # ===== TRAINING =====
    train_loss_sum, train_acc_sum, n_batches = 0.0, 0.0, 0
    pbar = tqdm(train_ds, total=n_train_steps, desc=f"Epoch {epoch+1} train")
    for batch in pbar:
        # Update LR (linear warmup + decay)
        lr_scale = update_learning_rates(
            global_step, total_steps, warmup_steps, initial_lrs, optimizers
        )
        # Train step
        loss, acc = train_step(
            model, batch, loss_fn, optimizers,
            var_groups_only, cfg.GRAD_CLIP, cfg.RDROP_ALPHA
        )
        train_loss_sum += loss.numpy()
        train_acc_sum  += acc.numpy()
        n_batches += 1
        # Log to TensorBoard every 50 steps
        if global_step % 50 == 0:
            with train_writer.as_default():
                tf.summary.scalar('loss', loss.numpy(), step=global_step)
                tf.summary.scalar('accuracy', acc.numpy(), step=global_step)
                tf.summary.scalar('learning_rate_head',
                                  optimizers[-1].learning_rate.numpy(),
                                  step=global_step)
                tf.summary.scalar('lr_scale', lr_scale, step=global_step)
        pbar.set_postfix({'loss': f'{loss.numpy():.4f}', 'acc': f'{acc.numpy():.4f}'})
        global_step += 1
    train_loss_avg = train_loss_sum / n_batches
    train_acc_avg  = train_acc_sum / n_batches
    # ===== VALIDATION =====
    val_loss_sum, val_acc_sum, vn = 0.0, 0.0, 0
    for batch in valid_ds:
        loss, acc = valid_step(model, batch, loss_fn)
        val_loss_sum += loss.numpy()
        val_acc_sum  += acc.numpy()
        vn += 1
    val_loss_avg = val_loss_sum / vn
    val_acc_avg  = val_acc_sum / vn
    # Log epoch metrics
    history['train_loss'].append(train_loss_avg)
    history['train_acc'].append(train_acc_avg)
    history['val_loss'].append(val_loss_avg)
    history['val_acc'].append(val_acc_avg)
    elapsed = time.time() - epoch_start_time
    print(f"\nEpoch {epoch+1} done in {elapsed:.1f}s")
    print(f"  train_loss={train_loss_avg:.4f}  train_acc={train_acc_avg:.4f}")
    print(f"  val_loss  ={val_loss_avg:.4f}  val_acc  ={val_acc_avg:.4f}")
    # TensorBoard epoch-level logs
    with train_writer.as_default():
        tf.summary.scalar('epoch_train_loss', train_loss_avg, step=epoch)
        tf.summary.scalar('epoch_train_acc', train_acc_avg, step=epoch)
        tf.summary.scalar('epoch_val_loss', val_loss_avg, step=epoch)
        tf.summary.scalar('epoch_val_acc', val_acc_avg, step=epoch)
    # Run F1 callback
    f1_callback.on_epoch_end(epoch)
    if f1_callback.stop_training:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

train_writer.flush()
print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

# Restore best weights
f1_callback.restore_best_weights()

## 15. Plot Training Curves

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss'])+1)

axes[0].plot(epochs_range, history['train_loss'], 'b-', label='train_loss', marker='o')
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='val_loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history['train_acc'], 'b-', label='train_acc', marker='o')
axes[1].plot(epochs_range, history['val_acc'], 'r-', label='val_acc', marker='s')

# Also plot macro F1 from callback history
val_f1s = [h['val_macro_f1'] for h in f1_callback.history]
axes[1].plot(epochs_range[:len(val_f1s)], val_f1s, 'g--', label='val_macro_f1', marker='^')

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Metric')
axes[1].set_title('Training & Validation Accuracy/F1')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0.85, color='gray', linestyle=':', alpha=0.5, label='Target 85%')

plt.tight_layout()
plt.savefig(f'{cfg.LOG_DIR}/training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Curves saved to {cfg.LOG_DIR}/training_curves.png")

## 16. Evaluation on Test Set
Final test untuk verifikasi target 85%.

In [ ]:
# Predict on test set
print("Predicting on test set...")
y_test_true, y_test_pred, y_test_probs = [], [], []

for batch_x, batch_y in tqdm(test_ds, desc='Test inference'):
    logits = model(batch_x, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    preds = np.argmax(probs, axis=-1)
    y_test_pred.extend(preds.tolist())
    y_test_true.extend(batch_y.numpy().tolist())
    y_test_probs.extend(probs.tolist())

y_test_true = np.array(y_test_true)
y_test_pred = np.array(y_test_pred)
y_test_probs = np.array(y_test_probs)

# Test metrics
test_acc = accuracy_score(y_test_true, y_test_pred)
test_macro_f1 = f1_score(y_test_true, y_test_pred, average='macro')
test_weighted_f1 = f1_score(y_test_true, y_test_pred, average='weighted')

print(f"\n{'='*70}\nTEST SET RESULTS\n{'='*70}")
print(f"Accuracy    : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"Macro F1    : {test_macro_f1:.4f}")
print(f"Weighted F1 : {test_weighted_f1:.4f}")
print(f"\nTarget Capstone (85%): {'✅ ACHIEVED' if test_acc >= 0.85 else '❌ NOT ACHIEVED'}")
print(f"{'='*70}")

In [ ]:
# Detailed classification report
target_names = [ID2LABEL[i] for i in range(cfg.NUM_CLASSES)]
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test_true, y_test_pred, target_names=target_names, digits=4))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_true, y_test_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalized per row)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(f'{cfg.LOG_DIR}/confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

## 17. Save Model — `.keras` & SavedModel
Dua format untuk safety: `.keras` (Keras 3 native) + SavedModel (TensorFlow standard).

In [ ]:
# Save in Keras format (.keras)
try:
    model.save(cfg.MODEL_KERAS)
    print(f"✓ Saved Keras format: {cfg.MODEL_KERAS}")
except Exception as e:
    print(f"⚠️ Keras save failed: {e}")
    print("Falling back to SavedModel only")

# Save in SavedModel format (more portable for production)
try:
    # Build with explicit signature for SavedModel export
    @tf.function(input_signature=[
        {
            'input_ids': tf.TensorSpec(shape=(None, cfg.MAX_LENGTH), dtype=tf.int32, name='input_ids'),
            'attention_mask': tf.TensorSpec(shape=(None, cfg.MAX_LENGTH), dtype=tf.int32, name='attention_mask'),
        }
    ])
    def serving_fn(inputs):
        logits = model(inputs, training=False)
        probs = tf.nn.softmax(logits, axis=-1)
        return {'logits': logits, 'probabilities': probs}
    tf.saved_model.save(
        model,
        cfg.SAVED_MODEL,
        signatures={'serving_default': serving_fn}
    )
    print(f"✓ Saved SavedModel format: {cfg.SAVED_MODEL}")
except Exception as e:
    print(f"⚠️ SavedModel save failed: {e}")

# Save test results & config
results = {
    'test_accuracy': float(test_acc),
    'test_macro_f1': float(test_macro_f1),
    'test_weighted_f1': float(test_weighted_f1),
    'best_val_macro_f1': float(f1_callback.best_f1),
    'best_val_epoch': int(f1_callback.best_epoch + 1),
    'config': {
        'model_name': cfg.MODEL_NAME,
        'num_classes': cfg.NUM_CLASSES,
        'max_length': cfg.MAX_LENGTH,
        'batch_size': cfg.BATCH_SIZE,
        'base_lr': cfg.BASE_LR,
        'epochs_run': len(history['train_loss']),
    },
    'label_mapping': ID2LABEL,
}
with open(f'{cfg.LOG_DIR}/results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"✓ Saved results: {cfg.LOG_DIR}/results.json")

## 18. Inference Code
Standalone function untuk prediksi text baru.

In [ ]:
def predict_emotion(text, model, tokenizer, max_length=128, id2label=None):
    """
    Predict emotion dari sebuah text.
    Args:
        text: string input
        model: trained EmotionClassifier
        tokenizer: BertTokenizer
        max_length: max tokens
        id2label: mapping label_id -> label_name
    Returns:
        dict with keys: label, label_id, confidence, all_scores
    """
    if id2label is None:
        id2label = ID2LABEL
    # Tokenize
    encoded = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='tf'
    )
    inputs = {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
    }
    # Predict
    logits = model(inputs, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]
    pred_id = int(np.argmax(probs))
    confidence = float(probs[pred_id])
    all_scores = {id2label[i]: float(p) for i, p in enumerate(probs)}
    return {
        'label': id2label[pred_id],
        'label_id': pred_id,
        'confidence': confidence,
        'all_scores': all_scores,
    }

# Test on example sentences
test_sentences = [
    "Aku sedih banget hari ini, semua kerjaan numpuk",
    "Bahagia banget, akhirnya lulus skripsi!",
    "Marah-marah terus, kerjaan ga selesai-selesai",
    "Takut banget besok presentasi di depan banyak orang",
    "Hari ini cuaca cerah",
]

print("=== INFERENCE EXAMPLES ===")
for sent in test_sentences:
    result = predict_emotion(sent, model, tokenizer, cfg.MAX_LENGTH)
    print(f"\nText: {sent}")
    print(f"  → {result['label']} (confidence: {result['confidence']:.2%})")
    # Top 3 scores
    sorted_scores = sorted(result['all_scores'].items(), key=lambda x: -x[1])
    for label, score in sorted_scores[:3]:
        print(f"     {label}: {score:.2%}")

## 19. Launch TensorBoard (Optional)
Visualisasi training curves di TensorBoard.

In [ ]:
# Uncomment untuk launch TensorBoard di Colab
# %load_ext tensorboard
# %tensorboard --logdir ./logs

# Atau kalau di local:
# !tensorboard --logdir ./logs --port 6006
print(f"To launch TensorBoard, run: tensorboard --logdir {cfg.LOG_DIR}")
print(f"Or in Colab: %tensorboard --logdir {cfg.LOG_DIR}")

## 20. Summary
Recap output dari training ini.

In [ ]:
print("=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)
print(f"\nDataset:")
print(f"  Train: {len(df_train)} | Valid: {len(df_valid)} | Test: {len(df_test)}")
print(f"  Classes: {target_names}")
print(f"\nModel: {cfg.MODEL_NAME} (Model Subclassing)")
_tp = sum(np.prod(v.shape) for v in model.trainable_variables)
_ap = sum(np.prod(v.shape) for v in model.variables)
print(f"  Total params: {_ap:,} (trainable: {_tp:,})")
print(f"  Custom Layer:    AttentionPoolingLayer ✓")
print(f"  Custom Loss:     FocalLossWithSmoothing ✓")
print(f"  Custom Callback: F1ScoreCallback ✓")
print(f"  Training Loop:   tf.GradientTape ✓")
print(f"  TensorBoard:     {cfg.LOG_DIR} ✓")
print(f"\nResults:")
print(f"  Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Test Macro F1 : {test_macro_f1:.4f}")
print(f"  Best Val F1   : {f1_callback.best_f1:.4f} (epoch {f1_callback.best_epoch+1})")
print(f"\nFiles saved:")
print(f"  Model (.keras): {cfg.MODEL_KERAS}")
print(f"  Model (SavedModel): {cfg.SAVED_MODEL}")
print(f"  Tokenizer: {cfg.TOKENIZER_DIR}")
print(f"  Logs/Results: {cfg.LOG_DIR}")
print(f"\nCapstone target (85%): {'✅ ACHIEVED' if test_acc >= 0.85 else '❌ NOT ACHIEVED'}")
print("=" * 70)